# GRPO Fine-tuning for verifiable tasks

You can run this notebook on any M-series Mac with a minimum of 24GB of RAM.

### What's in this notebook?

In this notebook you will learn how to fine-tune a Small Text-to-text Language Model for verifiable tasks using Group Relative Policy Optimization (GRPO).

GRPO is a Reinforcement Learning algorithm widely used by AI labs and practitioners to fine-tune models for easily verifiables tasks like

- Mathematical problem solving with numeric verification
- Code generation with unit test validation
- Structured output tasks (JSON, SQL) with schema validation
- Question answering with ground truth answers


We will cover
- Environment setup
- Data preparation
- Model training
- Local inference with your new model
- Model saving and exporting it into the format you need for **deployment**.

### Deployment options

LFM2.5 models are small and efficient, enabling deployment across a wide range of platforms:

<table align="left">
  <tr>
    <th>Deployment Target</th>
    <th>Use Case</th>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/android/android-quick-start-guide"><b>Android</b></a></td>
    <td>Mobile apps on Android devices</td>
  </tr>
  <tr>
    <td>📱 <a href="https://docs.liquid.ai/leap/edge-sdk/ios/ios-quick-start-guide"><b>iOS</b></a></td>
    <td>Mobile apps on iPhone/iPad</td>
  </tr>
  <tr>
    <td>🍎 <a href="https://docs.liquid.ai/docs/inference/mlx"><b>Apple Silicon Mac</b></a></td>
    <td>Local inference on Mac with MLX</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/llama-cpp"><b>llama.cpp</b></a></td>
    <td>Local deployments on any hardware</td>
  </tr>
  <tr>
    <td>🦙 <a href="https://docs.liquid.ai/docs/inference/ollama"><b>Ollama</b></a></td>
    <td>Local inference with easy setup</td>
  </tr>
  <tr>
    <td>🖥️ <a href="https://docs.liquid.ai/docs/inference/lm-studio"><b>LM Studio</b></a></td>
    <td>Desktop app for local inference</td>
  </tr>
  <tr>
    <td>⚡ <a href="https://docs.liquid.ai/docs/inference/vllm"><b>vLLM</b></a></td>
    <td>Cloud deployments with high throughput</td>
  </tr>
  <tr>
    <td>☁️ <a href="https://docs.liquid.ai/docs/inference/modal-deployment"><b>Modal</b></a></td>
    <td>Serverless cloud deployment</td>
  </tr>
  <tr>
    <td>🏗️ <a href="https://docs.liquid.ai/docs/inference/baseten-deployment"><b>Baseten</b></a></td>
    <td>Production ML infrastructure</td>
  </tr>
  <tr>
    <td>🚀 <a href="https://docs.liquid.ai/docs/inference/fal-deployment"><b>Fal</b></a></td>
    <td>Fast inference API</td>
  </tr>
</table>

### Need help building with our models and tools?
Join the Liquid AI Discord Community and ask.

<a href="https://discord.com/invite/liquid-ai"><img src="https://img.shields.io/discord/1385439864920739850?color=7289da&label=Join%20Discord&logo=discord&logoColor=white" alt="Join Discord"></a>

And now, let the fine tune begin!

## 📦 Installation & Setup

First, let's install all the required packages.

In [ ]:
%%capture
!pip install mlx-lm-lora

Let's now verify the packages are installed correctly

In [ ]:
from mlx_lm_lora.utils import from_pretrained, save_pretrained_merged, calculate_iters
from mlx_lm_lora.trainer.grpo_trainer import GRPOTrainingArgs, train_grpo
from mlx_lm_lora.trainer.datasets import CacheDataset, GRPODataset
from mlx_lm_lora.trainer.grpo_reward_functions import (
    r1_accuracy_reward_func,
    r1_int_reward_func,
    r1_strict_format_reward_func,
    r1_soft_format_reward_func,
    r1_count_xml
)
from datasets import load_dataset

from mlx_lm.tuner.utils import print_trainable_parameters, build_schedule

import mlx.optimizers as optim

In [ ]:
# Select a model to fine-tune from the list
lfm_models = [
    "LiquidAI/LFM2.5-1.2B-Instruct",
    "LiquidAI/LFM2.5-1.2B-JP",
    "LiquidAI/LFM2-8B-A1B",
    "LiquidAI/LFM2-2.6B-Exp",
    "LiquidAI/LFM2-2.6B",
    "LiquidAI/LFM2-700M",
    "LiquidAI/LFM2-350M",
]

# Model to fine-tune
model_id = "LiquidAI/LFM2-350M"
new_model_name = "lfm2-grpo"


ref_model, ref_tokenizer, adapter_file = from_pretrained(
    model=model_id,
    quantized_load={
        "bits": 8,
        "group_size": 64
    },
)

model, tokenizer, adapter_file = from_pretrained(
    model=model_id,
    new_adapter_path=f"./{new_model_name}",
    quantized_load={
        "bits": 8,
        "group_size": 64
    },
)
print_trainable_parameters(model)

## Load and transform the dataset

In this step, we load the [**mlx-community/simple-grpo**](https://huggingface.co/datasets/mlx-community/simple-grpo) dataset from the Hugging Face Hub using the `datasets` library.
This dataset focuses on **mathematical reasoning**, featuring problems that require step-by-step logical solutions.
By fine-tuning a model that does not yet exhibit strong reasoning capabilities, it can learn to **generate structured reasoning steps**, enhancing both the model's **accuracy** and **interpretability** on math-related tasks.

We will adapt our dataset to a conversational format using a custom system prompt, guiding the LLM to generate both step-by-step reasoning and the final answer.

In [ ]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant  "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process is enclosed strictly within <think> and </think> tags. "
    "After closing </think>, the assistant MUST provide the final answer in plain text."
)

def format(sample):
    system = SYSTEM_PROMPT
    prompt = sample["prompt"]
    answer = sample["answer"]
    return sample

train_dataset = load_dataset('mlx-community/simple-grpo', split='train[:50%]').map(format)
train_set = GRPODataset(train_dataset, tokenizer, prompt_key="prompt", answer_key="answer", system_key="system")

print(tokenizer.decode(train_set.process(train_set[0])[0]))

## Train model

In [ ]:
batch_size = 1 # Number of training samples processed in each forward/backward pass. A batch size of 1 minimizes memory usage.
epochs = 3 # Number of complete passes through the training dataset.

# Convert the desired number of epochs into the number of training iterations.
iters = calculate_iters(
    train_set,
    batch_size=batch_size,
    epochs=epochs
)

opt = optim.AdamW(
    learning_rate=1e-5, # Uses the dynamic learning-rate schedule defined above.
    betas=[0.9, 0.99], # Adam momentum coefficients. beta1 controls the moving average of gradients. beta2 controls the moving average of squared gradients.
    eps=1e-6, # Small numerical-stability constant used by AdamW.
    weight_decay=0.00, # Strength of weight decay regularization. 0.00 disables weight decay.
    bias_correction=False # Disables Adam's bias correction for the moving averages.
)

grpo_config = GRPOTrainingArgs(
    batch_size=1,
    iters=iters,
    gradient_accumulation_steps=32,
    val_batches=1,
    steps_per_report=32,
    steps_per_eval=10,
    steps_per_save=20,
    max_seq_length=1024,
    adapter_file=adapter_file,
    grad_checkpoint=True,
    group_size=16, # Number of candidate completions generated for each prompt. GRPO compares rewards within this group to estimate advantages.
    beta=0.05, # Strength of the KL regularization against the reference model. Higher values keep the policy closer to the reference model.
    epsilon=0.1, # Lower PPO-style clipping range for policy updates. Limits how much the probability ratio can change in one update.
    epsilon_high=0.1, # Upper clipping range for asymmetric policy-ratio clipping. Allows a different upper bound than the lower epsilon value.
    max_completion_length=512, # Maximum number of tokens the model may generate for each completion.
    reference_model_path=model_id, # Path/ID of the model used as the reference model for GRPO.
    temperature=0.7, # Sampling temperature used when generating candidate completions. Lower values make generations more deterministic.
    top_p=0.95, # Nucleus sampling: only samples from the smallest set of tokens whose cumulative probability reaches 95%.
    top_k=20, # Restricts sampling to the 20 highest-probability tokens at each generation step.
    min_p=0, # Minimum probability threshold relative to the most likely token. 0 disables min-p filtering.
    grpo_loss_type="dr_grpo", # GRPO objective variant. Available options: "grpo", "bnpo", or "dr_grpo".
    reward_weights=None, # Optional weights for combining multiple reward functions. None uses the trainer's default weighting behavior.
    importance_sampling_level="token", # Determines how importance-sampling ratios are calculated. "token" computes ratios per token, "sequence" at the sequence level, and None disables the corresponding importance-sampling correction.
)

train_grpo(
    model=model, # Policy model whose parameters/adapters will be trained.
    tokenizer=tokenizer, # Tokenizer used to encode prompts and decode generated completions.
    ref_model=ref_model.freeze(), # Frozen reference model used to regularize the policy and limit how far it moves away from the original model during training.
    args=grpo_config,
    optimizer=opt, # AdamW optimizer used to update the trainable model parameters.
    train_dataset=CacheDataset(train_set),
    val_dataset=None,
    reward_funcs=[
        r1_accuracy_reward_func,
        r1_int_reward_func,
        r1_strict_format_reward_func,
        r1_soft_format_reward_func,
        r1_count_xml,
    ],
)

## Save merged model

Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

In [ ]:
print("\n🔄 Merging and save LoRA weights...")
save_pretrained_merged(
    model=model, # Trained model.
    tokenizer=tokenizer, # Tokenizer saved alongside the model.
    save_path=new_model_name, # Directory/name for the final merged model.
    de_quantize=True, # Convert quantized weights back to regular weights when saving. You have to turn it to false when qat is enabled.
    remove_adapters=True # Merge/remove adapter structure so the result is a standalone model rather than a base model that requires separate adapter files.
)
print(f"💾 SFT Merged model saved to: {grpo_config.adapter_file}")